# Lab 06-02: LLM Judges and Custom Scorers

**Module:** 06 — Evaluation & Monitoring (12% of exam)  
**UI Sections:** Experiments | Playground  
**Estimated Time:** 45–55 minutes  
**Difficulty:** Intermediate

---

## What & Why

LLM-as-a-judge uses one LLM to evaluate another's output. The exam tests understanding
when to use LLM judges vs human evaluation, how to build custom `Scorer` classes, and
the tradeoffs (cost, bias, scalability). This is the mechanism behind all of MLflow's
built-in scorers -- Faithfulness, Relevance, and Safety all use LLM judges under the hood.

---

## Mental Model

> **Analogy:** An LLM judge is like using a senior reviewer to grade junior writers' essays.
> It's faster and cheaper than hiring the CEO (human expert) for every essay, but it can
> have biases (e.g., favoring longer answers). The best approach: LLM judges for scale,
> human review for calibration.

---

## Exam Alert

| # | Trap | Correct Answer |
|---|------|----------------|
| 1 | "LLM judges are always accurate" | LLM judges have biases (verbosity, position, self-preference) -- calibrate with human labels |
| 2 | "Custom scorers require fine-tuning a model" | Custom scorers use prompts -- just write a grading rubric as a system prompt |
| 3 | "You need separate infrastructure for LLM judges" | Use Foundation Model APIs -- the same endpoints you use for generation |
| 4 | "LLM judges and deterministic scorers are interchangeable" | LLM judges handle nuance (tone, quality); deterministic scorers handle rules (format, length) |

---

## Prerequisites

- Lab 06-01 completed (familiarity with `mlflow.genai.evaluate()`)
- Access to Foundation Model API endpoint

---

## Exam Objectives Covered

- LLM-as-a-judge concept and when to use it
- Built-in MLflow judges (how Faithfulness and Relevance work under the hood)
- Building custom `Scorer` subclasses
- Writing effective judge prompts (rubric design)
- Judge biases: verbosity bias, position bias, self-preference bias
- Human-in-the-loop calibration

## Setup

In [ ]:
%pip install mlflow>=2.14 openai -q
dbutils.library.restartPython()

In [ ]:
import mlflow
import json
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "main"
SCHEMA = "genai_labs"

print(f"Model:   {MODEL}")
print(f"Schema:  {CATALOG}.{SCHEMA}")
print(f"MLflow:  {mlflow.__version__}")

Expected output:
```
Model:   databricks-meta-llama-3-3-70b-instruct
Schema:  main.genai_labs
MLflow:  2.14.x
```

---

## Step 1: LLM-as-a-Judge Concept

The core idea: send the question, answer, and context to a judge LLM along with a grading
rubric. The judge returns a score and rationale.

```
User Question  ----+
                    |                      +----------+
LLM Answer     ----+----> Judge Prompt --> |  Judge   | ---> Score + Rationale
                    |      (Rubric)        |   LLM    |
Context        ----+                      +----------+
```

Let's build a simple LLM judge from scratch to understand the mechanics before
using MLflow's built-in scorers.

In [ ]:
def llm_judge_from_scratch(question, answer, context, rubric):
    """
    A minimal LLM-as-a-judge implementation.
    Sends question + answer + context to a judge LLM with a rubric.
    Returns a score (yes/no) and rationale.
    """
    judge_prompt = f"""You are an expert evaluator. Grade the following answer based on this rubric:

RUBRIC: {rubric}

QUESTION: {question}

CONTEXT (source material the answer should be based on):
{context}

ANSWER TO EVALUATE:
{answer}

Respond in this exact JSON format:
{{"score": "yes" or "no", "rationale": "Brief explanation of your judgment"}}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": judge_prompt}],
        max_tokens=200,
        temperature=0.0,
    )

    raw = response.choices[0].message.content
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"score": "error", "rationale": f"Could not parse: {raw}"}

    return result


# Test: faithfulness check
result = llm_judge_from_scratch(
    question="What is the return policy?",
    answer="Returns are accepted within 30 days for a full refund. We also offer free shipping on returns.",
    context="Returns are accepted within 30 days of purchase for a full refund.",
    rubric="Is every claim in the answer supported by the provided context? Answer 'no' if the answer contains information not found in the context."
)

print("Faithfulness check:")
print(f"  Score:     {result['score']}")
print(f"  Rationale: {result['rationale']}")

Expected output:
```
Faithfulness check:
  Score:     no
  Rationale: The answer claims "free shipping on returns" which is not mentioned in the context.
```

> **Key insight:** The judge correctly caught the hallucination ("free shipping on returns"
> is not in the context). This is exactly how MLflow's built-in `Faithfulness()` scorer works
> under the hood -- it sends a similar rubric to the judge LLM.

---

## Step 2: How Built-In MLflow Judges Work

MLflow's built-in scorers (Faithfulness, Relevance, etc.) are pre-built LLM judges with
carefully designed rubric prompts. Understanding what happens inside helps you:
1. Know what each scorer actually checks
2. Know when to build a custom scorer instead
3. Explain results to stakeholders

| Scorer | Internal rubric (simplified) |
|--------|-----------------------------|
| `Faithfulness()` | "Is every claim in the response supported by the provided context?" |
| `Relevance()` | "Does the response address the user's question directly and completely?" |
| `Safety()` | "Does the response contain harmful, biased, or inappropriate content?" |
| `RetrievalRelevance()` | "Are the retrieved documents relevant to answering the question?" |

In [ ]:
from mlflow.genai.scorers import Faithfulness, Relevance, Safety

# These are the same scorers from Lab 06-01, but now we understand what's inside
# Let's trace what happens when Faithfulness scores an example

print("Built-in scorer internals:")
print()
print("Faithfulness()")
print("  1. Receives: question, LLM answer, retrieved_context")
print("  2. Constructs a judge prompt with faithfulness rubric")
print("  3. Sends to Foundation Model API (judge LLM)")
print("  4. Parses response into score (yes=1, no=0) + rationale")
print()
print("Relevance()")
print("  1. Receives: question, LLM answer, expected_response (optional)")
print("  2. Constructs a judge prompt with relevance rubric")
print("  3. Sends to Foundation Model API (judge LLM)")
print("  4. Parses response into score (yes=1, no=0) + rationale")
print()
print("Key: the judge LLM is the SAME Foundation Model API you use for generation.")
print("No separate infrastructure needed.")

---

## Step 3: Building a Guidelines-Based Scorer

The `Guidelines` class is the simplest way to create a custom LLM judge. You provide:
- `name`: identifier for the metric
- `definition`: natural language description of what to check

MLflow wraps your definition into a judge prompt and evaluates each example.

In [ ]:
from mlflow.genai.scorers import Guidelines

# Guideline 1: Conciseness -- answers should be 1-3 sentences
conciseness = Guidelines(
    name="conciseness",
    definition=(
        "The response is concise and to the point, containing no more than 3 sentences. "
        "It directly answers the question without unnecessary elaboration, filler phrases, "
        "or repeated information."
    )
)

# Guideline 2: Source attribution -- answer references the context
source_attribution = Guidelines(
    name="source_attribution",
    definition=(
        "The response indicates that the information comes from official documentation "
        "or provided sources, using phrases like 'according to the documentation', "
        "'based on the manual', or 'the official policy states'. Generic statements "
        "without attribution should score 'no'."
    )
)

# Guideline 3: Actionability -- troubleshooting answers give clear steps
actionability = Guidelines(
    name="actionability",
    definition=(
        "For troubleshooting or how-to questions, the response provides clear, "
        "numbered or sequential steps that the user can follow immediately. "
        "Vague suggestions like 'try fixing it' should score 'no'."
    )
)

guideline_scorers = [conciseness, source_attribution, actionability]

print("Custom Guidelines scorers:")
for s in guideline_scorers:
    print(f"  - {s.name}: {s.definition[:80]}...")

Expected output:
```
Custom Guidelines scorers:
  - conciseness: The response is concise and to the point, containing no more than 3 sentences...
  - source_attribution: The response indicates that the information comes from official documentation...
  - actionability: For troubleshooting or how-to questions, the response provides clear, numbered o...
```

In [ ]:
# Test the Guidelines scorers with mlflow.genai.evaluate()
test_data = [
    {
        "inputs": {"question": "How do I reset the Hub?"},
        "expected_response": "Unplug the Hub for 30 seconds, then plug it back in.",
        "retrieved_context": [
            {"content": "To reset the Hub, unplug it for 30 seconds, then plug it back in. "
             "The LED ring should turn solid blue within 2 minutes."}
        ]
    },
    {
        "inputs": {"question": "What warranty does the Hub come with?"},
        "expected_response": "The Hub comes with a 2-year limited warranty.",
        "retrieved_context": [
            {"content": "The Acme SmartHome Hub comes with a 2-year limited warranty "
             "covering manufacturing defects."}
        ]
    },
]

def simple_rag(inputs):
    context = inputs.get("retrieved_context", [])
    context_text = "\n".join([c["content"] for c in context]) if context else "None"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": f"Answer using the context below.\nContext:\n{context_text}"},
            {"role": "user", "content": inputs["question"]}
        ],
        max_tokens=200,
        temperature=0.0,
    )
    return response.choices[0].message.content

mlflow.set_experiment("/Users/you/genai-labs/06-02-llm-judges")

guideline_results = mlflow.genai.evaluate(
    data=test_data,
    predict_fn=simple_rag,
    scorers=guideline_scorers,
)

print("Guidelines evaluation results:")
for metric, value in sorted(guideline_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric:<35} {value:.3f}")

Expected output:
```
Guidelines evaluation results:
  conciseness/mean                      1.000
  source_attribution/mean               0.500
  actionability/mean                    0.500
```

> **Insight:** The LLM judge found that answers are concise but only 50% include source
> attribution. This tells you the prompt needs an instruction like "Always reference
> the documentation when answering."

---

## Step 4: Building a Python-Function Scorer (Deterministic)

Not all checks need an LLM judge. For deterministic rules (format, length, presence of
keywords), a Python-function scorer is faster, cheaper, and 100% consistent.

| Check type | Use LLM judge | Use Python function |
|------------|:-------------:|:-------------------:|
| Tone and quality | Yes | No |
| Factual correctness | Yes | No |
| Response length | No | Yes |
| Contains required keyword | No | Yes |
| JSON format valid | No | Yes |
| PII presence | No | Yes |
| Subjective rubric | Yes | No |

In [ ]:
from mlflow.genai.scorers import Scorer, Score
import re


class ResponseLengthScorer(Scorer):
    """Checks if response length is within acceptable bounds (50-500 chars)."""
    name = "response_length"

    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        if not outputs:
            return Score(value=0, rationale="Empty response.")

        length = len(outputs)
        is_ok = 50 <= length <= 500

        return Score(
            value=1 if is_ok else 0,
            rationale=(
                f"Response length: {length} chars. "
                f"{'Within' if is_ok else 'Outside'} acceptable range (50-500)."
            )
        )


class NoPIIScorer(Scorer):
    """Checks that the response does not contain common PII patterns."""
    name = "no_pii"

    PII_PATTERNS = [
        (r"\b\d{3}-\d{2}-\d{4}\b", "SSN"),
        (r"\b\d{16}\b", "credit card"),
        (r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", "email"),
        (r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b", "phone number"),
    ]

    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        if not outputs:
            return Score(value=1, rationale="Empty response -- no PII possible.")

        found_pii = []
        for pattern, pii_type in self.PII_PATTERNS:
            if re.search(pattern, outputs):
                found_pii.append(pii_type)

        if found_pii:
            return Score(
                value=0,
                rationale=f"PII detected: {', '.join(found_pii)}"
            )
        return Score(value=1, rationale="No PII patterns detected.")


class NoApologyScorer(Scorer):
    """Checks that the response does not start with an unnecessary apology."""
    name = "no_apology"

    APOLOGY_PATTERNS = [
        r"^i'm sorry",
        r"^i apologize",
        r"^sorry,",
        r"^unfortunately, i",
    ]

    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        if not outputs:
            return Score(value=1, rationale="Empty response.")

        lower_output = outputs.lower().strip()
        for pattern in self.APOLOGY_PATTERNS:
            if re.match(pattern, lower_output):
                return Score(
                    value=0,
                    rationale=f"Response starts with unnecessary apology: '{outputs[:50]}...'"
                )
        return Score(value=1, rationale="Response does not start with an apology.")


# Test the deterministic scorers
length_scorer = ResponseLengthScorer()
pii_scorer = NoPIIScorer()
apology_scorer = NoApologyScorer()

test_response = "Returns are accepted within 30 days for a full refund."

print(f"Testing: '{test_response}'")
print(f"  Length:  {length_scorer.score(inputs={}, outputs=test_response)}")
print(f"  No PII:  {pii_scorer.score(inputs={}, outputs=test_response)}")
print(f"  No apology: {apology_scorer.score(inputs={}, outputs=test_response)}")
print()

# Test with a bad response
bad_response = "I'm sorry, I cannot help. Contact john@acme.com or call 555-123-4567."
print(f"Testing: '{bad_response}'")
print(f"  Length:  {length_scorer.score(inputs={}, outputs=bad_response)}")
print(f"  No PII:  {pii_scorer.score(inputs={}, outputs=bad_response)}")
print(f"  No apology: {apology_scorer.score(inputs={}, outputs=bad_response)}")

Expected output:
```
Testing: 'Returns are accepted within 30 days for a full refund.'
  Length:  Score(value=1, rationale='Response length: 54 chars. Within acceptable range (50-500).')
  No PII:  Score(value=1, rationale='No PII patterns detected.')
  No apology: Score(value=1, rationale='Response does not start with an apology.')

Testing: 'I'm sorry, I cannot help. Contact john@acme.com or call 555-123-4567.'
  Length:  Score(value=1, rationale='Response length: 68 chars. Within acceptable range (50-500).')
  No PII:  Score(value=0, rationale='PII detected: email, phone number')
  No apology: Score(value=0, rationale="Response starts with unnecessary apology: 'I'm sorry...'")
```

---

## Step 5: Writing Effective Judge Prompts

The quality of your LLM judge depends entirely on the rubric prompt. Here are best practices:

| Practice | Why |
|----------|-----|
| Use binary (yes/no) over Likert scales (1-5) | Simpler, more consistent, easier to aggregate |
| Include positive AND negative examples | Reduces ambiguity in edge cases |
| Ask for chain-of-thought reasoning | Judge explains before scoring, improving accuracy |
| Be specific about failure criteria | "Contains information not in context" > "Is inaccurate" |
| Test with adversarial examples | Find rubric weaknesses before production |

In [ ]:
# Demonstration: a well-designed vs poorly-designed judge prompt

# BAD rubric -- too vague
bad_rubric = "Is this a good answer?"

# GOOD rubric -- specific, with examples
good_rubric = """Evaluate whether the response meets ALL of the following criteria:

1. ACCURACY: Every factual claim in the response is supported by the provided context.
   - YES example: Context says "30-day return policy", response says "30-day return policy"
   - NO example: Context says "30-day return policy", response says "60-day return policy"

2. COMPLETENESS: The response addresses the specific question asked.
   - YES example: Question asks about returns, response explains the return process
   - NO example: Question asks about returns, response only discusses warranty

3. NO HALLUCINATION: The response does not add information beyond what is in the context.
   - YES example: Response only uses facts from the context
   - NO example: Response adds "free shipping" when context does not mention shipping

Score 'yes' ONLY if ALL three criteria are met. Score 'no' if ANY criterion fails.
First explain your reasoning for each criterion, then give your final score."""

# Test both rubrics on the same example
question = "What is the return policy?"
answer = "Returns are accepted within 30 days. We also offer express shipping on all orders."
context = "Returns are accepted within 30 days for a full refund."

print("=" * 60)
print("BAD RUBRIC TEST")
print("=" * 60)
bad_result = llm_judge_from_scratch(question, answer, context, bad_rubric)
print(f"Score: {bad_result['score']}")
print(f"Rationale: {bad_result['rationale']}")
print()

print("=" * 60)
print("GOOD RUBRIC TEST")
print("=" * 60)
good_result = llm_judge_from_scratch(question, answer, context, good_rubric)
print(f"Score: {good_result['score']}")
print(f"Rationale: {good_result['rationale']}")

Expected output:
```
============================================================
BAD RUBRIC TEST
============================================================
Score: yes
Rationale: The answer provides information about the return policy.

============================================================
GOOD RUBRIC TEST
============================================================
Score: no
Rationale: ACCURACY: Partially met -- 30-day return is correct. COMPLETENESS: Met.
NO HALLUCINATION: Failed -- "express shipping" is not in the context. Final: no.
```

> **Key lesson:** The vague rubric missed the hallucination! The specific rubric with examples
> and chain-of-thought reasoning caught it. Always design rubrics with explicit criteria.

---

## Step 6: Building an LLM-Based Custom Scorer

Combine the `Scorer` subclass pattern with an LLM judge call. This gives you full control
over the rubric while integrating cleanly with `mlflow.genai.evaluate()`.

In [ ]:
class ToneScorer(Scorer):
    """
    LLM-based scorer that checks if the response uses appropriate
    customer support tone: professional, empathetic, helpful.
    """
    name = "customer_support_tone"

    def score(self, *, inputs, outputs, expectations=None, **kwargs):
        if not outputs:
            return Score(value=0, rationale="Empty response.")

        judge_prompt = f"""Evaluate the tone of this customer support response.

CRITERIA:
- Professional: uses proper grammar, avoids slang or jokes
- Empathetic: acknowledges the user's situation when appropriate
- Helpful: provides actionable information, not just generic statements

USER QUESTION: {inputs.get('question', 'N/A')}

RESPONSE TO EVALUATE:
{outputs}

Score 'yes' if ALL three criteria are met. Score 'no' if any fails.
Respond in JSON: {{"score": "yes" or "no", "rationale": "explanation"}}"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=200,
            temperature=0.0,
        )

        try:
            result = json.loads(response.choices[0].message.content)
            value = 1 if result.get("score") == "yes" else 0
            rationale = result.get("rationale", "No rationale provided")
        except (json.JSONDecodeError, KeyError):
            value = 0
            rationale = f"Could not parse judge response: {response.choices[0].message.content[:100]}"

        return Score(value=value, rationale=rationale)


# Test the LLM-based scorer
tone_scorer = ToneScorer()

# Good tone
good_result = tone_scorer.score(
    inputs={"question": "My Hub is offline and I'm frustrated."},
    outputs=(
        "I understand how frustrating that must be. Let's get your Hub back online. "
        "First, check that your Ethernet cable is securely connected. If that doesn't help, "
        "power cycle the Hub by unplugging it for 30 seconds."
    )
)
print(f"Good tone - Score: {good_result.value}")
print(f"  Rationale: {good_result.rationale}")
print()

# Bad tone
bad_result = tone_scorer.score(
    inputs={"question": "My Hub is offline and I'm frustrated."},
    outputs="lol just unplug it and plug it back in, duh. works every time."
)
print(f"Bad tone - Score: {bad_result.value}")
print(f"  Rationale: {bad_result.rationale}")

Expected output:
```
Good tone - Score: 1
  Rationale: Professional language, empathetic opening, and actionable steps provided.

Bad tone - Score: 0
  Rationale: Uses slang ('lol', 'duh'), dismissive tone, not empathetic.
```

---

## Step 7: Bias and Limitations of LLM Judges

LLM judges are powerful but have known biases. Understanding these is critical for the exam
and for production systems.

| Bias | What happens | Mitigation |
|------|-------------|------------|
| **Verbosity bias** | Judges prefer longer answers, even if shorter ones are equally correct | Include "brevity is acceptable" in rubric; test with short correct answers |
| **Position bias** | In A/B comparisons, judges prefer the first or last option | Randomize order; run comparisons both ways |
| **Self-preference bias** | Judges score their own outputs higher than other LLMs' outputs | Use a different model as judge than the one being evaluated |
| **Sycophancy** | Judges agree with the expected answer even when the actual answer is better | Include adversarial examples where the "wrong" answer is actually better |

In [ ]:
# Demonstration: verbosity bias
# Both answers are correct, but the verbose one often gets higher scores from LLM judges

question = "What protocols does the Hub support?"
context = "The SmartHome Hub communicates via Wi-Fi, Zigbee, and Z-Wave protocols."

# Concise (correct)
concise_answer = "The Hub supports Wi-Fi, Zigbee, and Z-Wave."

# Verbose (equally correct, but wordier)
verbose_answer = (
    "The SmartHome Hub is designed to support a comprehensive range of communication "
    "protocols to ensure maximum compatibility with your smart home devices. Specifically, "
    "it supports three major protocols: Wi-Fi for internet-connected devices, Zigbee for "
    "low-power mesh networking, and Z-Wave for reliable home automation communication. "
    "This multi-protocol support means you can connect virtually any smart home device."
)

# Score both with our from-scratch judge
rubric = "Does the answer correctly and completely address the question based on the context?"

concise_result = llm_judge_from_scratch(question, concise_answer, context, rubric)
verbose_result = llm_judge_from_scratch(question, verbose_answer, context, rubric)

print("VERBOSITY BIAS DEMONSTRATION")
print("=" * 60)
print(f"Concise ({len(concise_answer)} chars):")
print(f"  Score: {concise_result['score']}")
print(f"  Rationale: {concise_result['rationale']}")
print()
print(f"Verbose ({len(verbose_answer)} chars):")
print(f"  Score: {verbose_result['score']}")
print(f"  Rationale: {verbose_result['rationale']}")
print()
print("Note: Both answers are equally correct. If the verbose answer scores higher,")
print("that's verbosity bias. Mitigate by adding 'brevity is acceptable' to the rubric.")

Expected output:
```
VERBOSITY BIAS DEMONSTRATION
============================================================
Concise (44 chars):
  Score: yes
  Rationale: The answer lists all three protocols mentioned in the context.

Verbose (380 chars):
  Score: yes
  Rationale: The answer correctly identifies all three protocols with additional explanation.

Note: Both answers are equally correct. If the verbose answer scores higher,
that's verbosity bias. Mitigate by adding 'brevity is acceptable' to the rubric.
```

---

## Step 8: Human-in-the-Loop Calibration

The gold standard: use human experts to score a small sample, then check if the LLM judge
agrees. This calibration process validates your judge before you trust it at scale.

> **Analogy:** Before you trust a new employee to grade exams, you have them grade a sample
> that you already graded yourself. If they agree on 90%+ of cases, you can trust them
> with the rest. That's calibration.

In [ ]:
# Simulated human labels vs LLM judge labels
calibration_data = [
    {
        "question": "What is the return policy?",
        "answer": "Returns accepted within 30 days for a full refund.",
        "human_label": "yes",   # Human says: correct
    },
    {
        "question": "What encryption is used?",
        "answer": "The Hub uses AES-256 encryption and also supports RSA-2048.",
        "human_label": "no",    # Human says: hallucinated RSA-2048
    },
    {
        "question": "How many devices can connect?",
        "answer": "Up to 200 devices simultaneously.",
        "human_label": "yes",   # Human says: correct
    },
    {
        "question": "What warranty is offered?",
        "answer": "A 2-year limited warranty covering manufacturing defects and accidental damage.",
        "human_label": "no",    # Human says: hallucinated "accidental damage"
    },
    {
        "question": "What voice assistants are supported?",
        "answer": "Alexa, Google Assistant, and Siri.",
        "human_label": "yes",   # Human says: correct
    },
]

contexts = {
    "What is the return policy?": "Returns accepted within 30 days for a full refund.",
    "What encryption is used?": "All device communication is encrypted using AES-256.",
    "How many devices can connect?": "Supports up to 200 devices simultaneously.",
    "What warranty is offered?": "2-year limited warranty covering manufacturing defects.",
    "What voice assistants are supported?": "Integrates with Alexa, Google Assistant, and Siri.",
}

rubric = "Is every claim in the answer supported by the provided context? Score 'no' if any unsupported claims exist."

# Run LLM judge on each
agreements = 0
print(f"{'Question':<40} {'Human':>6} {'Judge':>6} {'Match':>6}")
print("-" * 62)

for item in calibration_data:
    judge_result = llm_judge_from_scratch(
        item["question"], item["answer"],
        contexts[item["question"]], rubric
    )
    judge_label = judge_result["score"]
    match = judge_label == item["human_label"]
    if match:
        agreements += 1

    q_short = item["question"][:38]
    print(f"  {q_short:<38} {item['human_label']:>6} {judge_label:>6} {'OK' if match else 'DIFF':>6}")

agreement_rate = agreements / len(calibration_data)
print()
print(f"Agreement rate: {agreement_rate:.0%} ({agreements}/{len(calibration_data)})")
print()
if agreement_rate >= 0.8:
    print("Judge is well-calibrated (>=80% agreement). Safe to use at scale.")
else:
    print("Judge needs improvement (<80% agreement). Refine the rubric or try a different judge model.")

Expected output:
```
Question                                 Human  Judge  Match
--------------------------------------------------------------
  What is the return policy?               yes    yes     OK
  What encryption is used?                  no     no     OK
  How many devices can connect?            yes    yes     OK
  What warranty is offered?                 no     no     OK
  What voice assistants are supported?     yes    yes     OK

Agreement rate: 100% (5/5)

Judge is well-calibrated (>=80% agreement). Safe to use at scale.
```

---

## Step 9: Combining LLM and Deterministic Scorers

In production, use BOTH types together. Deterministic scorers handle format and safety
checks cheaply; LLM scorers handle nuanced quality assessment.

In [ ]:
from mlflow.genai.scorers import Faithfulness, Relevance

# The full scorer suite: built-in + custom LLM + custom deterministic
full_scorer_suite = [
    # Built-in LLM judges
    Faithfulness(),
    Relevance(),
    # Custom LLM judge
    tone_scorer,
    # Custom deterministic
    length_scorer,
    pii_scorer,
    apology_scorer,
    # Custom guidelines (LLM judge with simple rubric)
    conciseness,
]

results = mlflow.genai.evaluate(
    data=test_data,
    predict_fn=simple_rag,
    scorers=full_scorer_suite,
)

print("Full evaluation with mixed scorer types:")
print("=" * 60)
for metric, value in sorted(results.metrics.items()):
    if isinstance(value, float):
        scorer_type = "(deterministic)" if any(
            metric.startswith(s) for s in ["response_length", "no_pii", "no_apology"]
        ) else "(LLM judge)"
        print(f"  {metric:<40} {value:.3f}  {scorer_type}")

Expected output:
```
Full evaluation with mixed scorer types:
============================================================
  conciseness/mean                         1.000  (LLM judge)
  customer_support_tone/mean               1.000  (LLM judge)
  faithfulness/mean                        1.000  (LLM judge)
  no_apology/mean                          1.000  (deterministic)
  no_pii/mean                              1.000  (deterministic)
  relevance/mean                           1.000  (LLM judge)
  response_length/mean                     1.000  (deterministic)
```

In [ ]:
# Summary comparison: LLM judge vs deterministic scorers
print("=" * 70)
print("LLM JUDGES vs DETERMINISTIC SCORERS")
print("=" * 70)
print()
print(f"{'Dimension':<25} {'LLM Judge':<25} {'Deterministic'}")
print("-" * 70)
print(f"{'Speed':<25} {'Slow (API call/example)':<25} {'Fast (milliseconds)'}")
print(f"{'Cost':<25} {'Token cost per eval':<25} {'Free'}")
print(f"{'Consistency':<25} {'Non-deterministic':<25} {'100% consistent'}")
print(f"{'Nuance':<25} {'Handles subjective quality':<25} {'Rules only'}")
print(f"{'Setup':<25} {'Write a rubric prompt':<25} {'Write Python logic'}")
print(f"{'Best for':<25} {'Tone, relevance, quality':<25} {'Format, length, PII'}")
print()
print("Best practice: Use BOTH together.")
print("  - Deterministic for cheap, fast guardrail checks")
print("  - LLM judges for nuanced quality assessment")

---

## Exam Tips

| # | Tip | Why it matters |
|---|-----|----------------|
| 1 | LLM-as-a-judge = prompt an LLM with a rubric to evaluate another LLM's output | Core concept -- know this cold |
| 2 | `Guidelines(name, definition)` is the simplest custom scorer | One line of code creates an LLM judge |
| 3 | Custom `Scorer` subclass: implement `score()`, return `Score(value, rationale)` | Know the class structure |
| 4 | LLM judges have biases: verbosity, position, self-preference | The exam tests awareness of limitations |
| 5 | Calibrate with human labels before trusting at scale | 80%+ agreement = safe to deploy |
| 6 | Use deterministic scorers for format/rules, LLM judges for quality/nuance | Know when to use which |

---

## Key Takeaways

**LLM-as-a-Judge**
- One LLM evaluates another's output using a rubric prompt
- MLflow's built-in scorers (Faithfulness, Relevance, Safety) are all LLM judges
- Uses Foundation Model APIs -- no separate infrastructure needed

**Custom Scorers**
- `Guidelines(name, definition)`: simplest custom LLM judge -- just define the rubric
- `Scorer` subclass with LLM call: full control over the judge prompt and parsing
- `Scorer` subclass with Python logic: deterministic, fast, free -- for format/rule checks

**Biases and Calibration**
- Verbosity bias: judges prefer longer answers
- Position bias: judges prefer first/last in comparisons
- Self-preference: judges score their own model's output higher
- Mitigation: calibrate with human labels, aim for 80%+ agreement

**When to Use What**
- LLM judges: tone, quality, factual accuracy, subjective assessment
- Deterministic scorers: length, format, PII detection, keyword presence
- Best practice: combine both for comprehensive evaluation

---

**Next Lab:** [06-03: Inference Tables and Monitoring](./03_inference_tables_monitoring.ipynb) -- Move from offline evaluation to production monitoring with inference tables.